# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas

Esta atividade foi construída com base nos slides da Aula 3, que destacam a **limpeza como a fundação invisível** de qualquer dashboard confiável. O objetivo aqui não é apenas "fazer funcionar", mas tomar decisões conscientes sobre tipos, valores ausentes, duplicidades, variáveis derivadas e exportação da base limpa. fileciteturn2file0

## Regras desta atividade
- Você deve **construir os códigos**.
- O notebook orienta os passos, mas não entrega as soluções prontas.
- Sempre que fizer uma decisão de limpeza, **documente o porquê** em comentário ou célula markdown.
- Ao final, exporte uma base limpa para uso nas próximas aulas.

## Dataset desta atividade
Arquivo bruto: `vendas_brasil_aula3_bruto.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para trabalhar com:
- leitura de dados
- tratamento de nulos
- identificação de infinitos
- exportação do resultado final

**Sugestão:** Pandas e NumPy já resolvem toda a atividade.


In [1]:
import pandas as pd
import numpy as np

## 2. Leitura da base bruta

Leia o arquivo `vendas_brasil_aula3_bruto.csv` em um DataFrame chamado `df`.

Depois:
1. Exiba as primeiras linhas
2. Exiba as últimas linhas
3. Observe visualmente se já existem sinais de sujeira


In [13]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

df.head()
df.tail()

,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao
223,1180,2024-03-11,SC,Loja Física,Telefonia,Smartphone X,C158,1,2563.44,415.42,NaN
224,1015,2024-02-26,RS,Online,Informática,Notebook Pro,C102,3,15011.49,NaN,cliente premium
225,1115,2024-07-29,MG,Loja Física,Informática,Notebook Pro,C144,3,12566.04,2879.76,NaN
226,1172,2024-11-11,ES,Loja Física,Informática,Notebook Pro,C128,3,13154.04,3408.22,entrega rápida
227,1209,2024-05-27,MG,Marketplace,Áudio,Caixa de Som,C121,2,810.2,282.52,NaN


## 3. Check-up inicial do dataset

Com base no checklist de um dataset "clean" apresentado na aula, faça um diagnóstico inicial da base. fileciteturn2file0

### Sua missão
Use comandos que permitam responder:
- Qual é o tamanho da base?
- Quais são os tipos atuais das colunas?
- Existem valores nulos?
- Há colunas com tipo inadequado?
- Há algo que pareça impedir análises confiáveis?


In [12]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

df.head()
df.tail()

,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao
223,1180,2024-03-11,SC,Loja Física,Telefonia,Smartphone X,C158,1,2563.44,415.42,NaN
224,1015,2024-02-26,RS,Online,Informática,Notebook Pro,C102,3,15011.49,NaN,cliente premium
225,1115,2024-07-29,MG,Loja Física,Informática,Notebook Pro,C144,3,12566.04,2879.76,NaN
226,1172,2024-11-11,ES,Loja Física,Informática,Notebook Pro,C128,3,13154.04,3408.22,entrega rápida
227,1209,2024-05-27,MG,Marketplace,Áudio,Caixa de Som,C121,2,810.2,282.52,NaN


## 4. Batalha 1 — A tirania dos tipos de dados

Nos slides, vimos que datas não podem ser tratadas como texto e que números em formato string quebram análises. fileciteturn2file0

### Tarefa
Converta corretamente, quando fizer sentido:
- `data`
- `receita`
- `lucro`
- `quantidade`

### Orientação
- Investigue valores estranhos antes de converter
- Pense no uso de `errors='coerce'`
- Registre em markdown ou comentário quais problemas encontrou


In [14]:
# Data
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# Receita
receita_limpa = (
    df['receita'].astype(str)
    .str.replace('R\$', '', regex=True)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

df['receita'] = pd.to_numeric(receita_limpa, errors='coerce')

# Lucro
df['lucro'] = pd.to_numeric(df['lucro'], errors='coerce')

# Quantidade
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

<>:7: SyntaxWarning: invalid escape sequence '\$'
<>:7: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_154/1864101398.py:7: SyntaxWarning: invalid escape sequence '\$'
  .str.replace('R\$', '', regex=True)


### Reflexão curta
Explique:
1. Por que deixar `data` como texto pode quebrar análises temporais?
2. Por que deixar `receita` como string pode distorcer cálculos?


## 5. Batalha 2 — O enigma dos valores ausentes

A aula reforça que valores ausentes não devem ser tratados automaticamente; a decisão precisa ser justificada. fileciteturn2file0

### Tarefa
1. Mapeie os valores ausentes por coluna
2. Identifique quais colunas críticas têm nulos
3. Defina uma estratégia para cada caso:
   - remover linhas?
   - preencher?
   - manter?
4. Justifique cada escolha

### Dica
Evite decisões automáticas sem análise de contexto.


In [15]:
df.isna().sum()

,0
pedido_id,0
data,224
uf,5
canal,5
categoria,5
produto,0
cliente_id,0
quantidade,0
receita,5
lucro,6


### Registro reflexivo
Escreva aqui sua justificativa para o tratamento dos nulos:
- Em quais colunas você removeu linhas?
- Em quais colunas você preencheu valores?
- Em quais situações decidiu manter o nulo?


## 6. Batalha 3 — O ataque dos clones (duplicidades)

A aula alerta que nem toda duplicidade é erro automático: pode haver compra legítima repetida ou dupla inserção do sistema. fileciteturn2file0

### Tarefa
1. Investigue se há linhas duplicadas
2. Analise se a duplicidade parece nociva para os KPIs
3. Escolha uma estratégia:
   - remover duplicidades totais?
   - remover apenas com base em certas colunas?
   - manter?
4. Justifique a decisão


In [17]:
df.duplicated().sum()

df = df.drop_duplicates()

## 7. Padronização de categorias

Os slides mostram que categorias duplicadas ou mal escritas podem distorcer rankings e filtros. fileciteturn2file0

### Tarefa
Inspecione e padronize, se necessário:
- `uf`
- `canal`
- `categoria`

### Pense em:
- maiúsculas e minúsculas
- espaços extras
- acentuação / variações
- categorias equivalentes escritas de formas diferentes


In [18]:
# UF
df['uf'] = df['uf'].str.upper().str.strip()

# Canal
df['canal'] = df['canal'].str.lower().str.strip()

# Categoria
df['categoria'] = df['categoria'].str.lower().str.strip()

## 8. Subindo de nível — Criação de variáveis derivadas

Depois da limpeza, é hora de enriquecer a base com variáveis úteis para análise. fileciteturn2file0

### Tarefa
Crie, se possível:
- `ano`
- `mes`
- `ano_mes`
- `margem_lucro`

### Atenção
A criação de `margem_lucro` exige cuidado com divisões por zero e possíveis valores infinitos.


In [19]:
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['ano_mes'] = df['data'].dt.to_period('M')

df['margem_lucro'] = df['lucro'] / df['receita']

# Tratar infinitos
df['margem_lucro'] = df['margem_lucro'].replace([np.inf, -np.inf], np.nan)

### Reflexão técnica
Explique:
1. O que pode acontecer ao calcular margem de lucro quando a receita é zero?
2. Como você decidiu tratar esse caso?


## 9. Seleção final — Menos é mais

A aula propõe que nem toda coluna precisa seguir para a base analítica final. fileciteturn2file0

### Tarefa
Crie um DataFrame final, por exemplo `df_clean` ou `df_dash`, contendo apenas as colunas estritamente necessárias para análises de negócio.

**Sugestão de raciocínio:**
- Quais colunas ajudam a responder perguntas de negócio?
- Quais colunas são operacionais, auxiliares ou dispensáveis?
- O que precisa existir para as próximas aulas de visualização e dashboard?


In [20]:
df_clean = df[
    ['data', 'ano', 'mes', 'ano_mes',
     'uf', 'canal', 'categoria',
     'receita', 'lucro', 'quantidade', 'margem_lucro']
].copy()

## 10. Validação final da base limpa

Antes de exportar, faça uma checagem final.

### Verifique:
- os tipos estão corretos?
- ainda há nulos problemáticos?
- ainda há duplicidades nocivas?
- existe algum infinito em `margem_lucro`?
- a base está pronta para ser usada em análises e dashboards?


In [21]:
df_clean.info()
df_clean.isna().sum()
df_clean.duplicated().sum()

np.isinf(df_clean['margem_lucro']).sum()

<class 'pandas.core.frame.DataFrame'>
Index: 221 entries, 0 to 224
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data          4 non-null      datetime64[ns]
 1   ano           4 non-null      float64       
 2   mes           4 non-null      float64       
 3   ano_mes       4 non-null      period[M]     
 4   uf            216 non-null    object        
 5   canal         216 non-null    object        
 6   categoria     216 non-null    object        
 7   receita       216 non-null    float64       
 8   lucro         215 non-null    float64       
 9   quantidade    221 non-null    int64         
 10  margem_lucro  206 non-null    float64       
dtypes: datetime64[ns](1), float64(5), int64(1), object(3), period[M](1)
memory usage: 20.7+ KB


np.int64(0)

## 11. Exportação da base "Clean"

Agora gere o arquivo final da base tratada.

### Tarefa
Exporte sua base limpa com o nome:

`vendas_brasil_aula3_clean.csv`

### Observação
Esse arquivo será o selo de qualidade do seu trabalho desta aula.


In [22]:
df_clean.to_csv('vendas_brasil_aula3_clean.csv', index=False)

## 12. Conclusão e registro reflexivo

Escreva um pequeno texto respondendo:

1. Quais foram os principais problemas de qualidade encontrados?
2. Qual decisão de limpeza foi mais difícil?
3. Que impacto essas falhas poderiam causar em um dashboard?
4. Por que a etapa de limpeza é considerada a "fundação invisível" do projeto?


Principais problemas encontrados:

Tipos incorretos (datas e valores numéricos como texto)
Valores nulos em colunas importantes
Categorias inconsistentes
Possíveis duplicidades

Decisão mais difícil:

Definir quando remover ou manter valores nulos, pois isso impacta diretamente os resultados

Impacto no dashboard:

Métricas incorretas (ex: receita errada)
Rankings distorcidos
Análises temporais inviáveis

Por que limpeza é a base invisível:

Sem dados confiáveis, qualquer visualização será enganosa
O usuário confia no dashboard — mesmo que esteja errado

## 13. Desafio extra (opcional)

Use sua base limpa para responder rapidamente:

- Qual UF concentra maior receita?
- Qual canal gera maior lucro?
- Existe algum mês com desempenho claramente acima dos demais?

Você pode responder com tabelas simples ou pequenos agrupamentos.


In [23]:
# UF com maior receita
df_clean.groupby('uf')['receita'].sum().sort_values(ascending=False)

# Canal com maior lucro
df_clean.groupby('canal')['lucro'].sum().sort_values(ascending=False)

# Receita por mês
df_clean.groupby('ano_mes')['receita'].sum().sort_values(ascending=False)

,receita
ano_mes,
2024-10,1146636.0
2024-04,453511.0
2024-09,399883.0
